# Module 07 — Chi-square & kiểm định phi tham số
Notebook RIÊNG cho Module 07 — tái lập cả 6 kiểm định trong bài: chi-square (2 bảng), Mann-Whitney U, Wilcoxon, Kruskal-Wallis, Friedman.

Tương ứng SPSS: `Analyze > Descriptive Statistics > Crosstabs` (Chi-square), `Analyze > Nonparametric Tests > Legacy Dialogs`.

In [ ]:
# Chạy ô này đầu tiên (Colab: bấm ▶). Không cần cài gì thêm.
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid"); plt.rcParams["figure.figsize"] = (7, 4)
pd.set_option("display.precision", 3)

def make_data(seed=2026, n=240):
    """Bộ dữ liệu GIẢ LẬP 'Lớp học 240 học sinh' — CÙNG một seed=2026 dùng
    xuyên suốt các module 03-08, khớp chính xác với lop_hoc_240.csv/.sav
    trong repo (không phải dữ liệu thật, nhưng tái lập được 100%)."""
    rng = np.random.default_rng(seed)
    school = rng.choice(list("ABC"), n, p=[.35, .35, .30])
    gender = rng.choice(["Nam", "Nữ"], n)
    method = rng.choice(["Truyền thống", "Dự án"], n)
    study_hours = np.clip(rng.gamma(4, 1.2, n), 0.5, 15).round(1)
    interest = rng.normal(0, 1, n) + 0.15 * (method == "Dự án")
    anxiety = rng.normal(0, 1, n) - 0.25 * interest
    def likert(lat, load, noise=0.7):
        return np.clip(np.round(3 + load * lat + rng.normal(0, noise, n)), 1, 5).astype(int)
    d = pd.DataFrame({"id": np.arange(1, n + 1), "school": school, "gender": gender, "method": method,
                      "study_hours": study_hours})
    for k, (lat, load) in enumerate([(interest, .8), (interest, .7), (interest, .75)], 1):
        d[f"h{k}"] = likert(lat, load)
    for k, (lat, load) in enumerate([(anxiety, .8), (anxiety, .75), (anxiety, .7)], 1):
        d[f"a{k}"] = likert(lat, load)
    eff = d.school.map({"A": 3, "B": 0, "C": -3}).to_numpy()
    d["pretest"] = (rng.normal(60, 10, n) + eff).round(1)
    d["posttest"] = np.clip(d.pretest + 3 + 5 * (method == "Dự án") + rng.normal(0, 6, n), 0, 100).round(1)
    d["math"] = np.clip(35 + 2.5 * study_hours + 4 * interest - 3 * anxiety + eff + rng.normal(0, 6, n), 0, 100).round(1)
    d.loc[6, "study_hours"] = 48.0
    d.loc[[11, 57, 130], "h2"] = np.nan
    d["h2"] = d["h2"].astype("Int64")
    return d

df = make_data()
print("Đã tạo df:", df.shape)

## 1. Chi-square: giới tính × phương pháp dạy (bảng 2×2, dùng Yates continuity correction)

🎯 **Phương pháp này trả lời câu hỏi gì?** Chi-square kiểm tra: hai biến ĐỊNH DANH có ĐỘC LẬP với nhau hay không — nói cách khác, biết giá trị của biến này có giúp đoán giá trị biến kia TỐT HƠN so với đoán ngẫu nhiên hay không. Giả thuyết không (H0): 2 biến độc lập, tức là tần số QUAN SÁT được ở mỗi ô của bảng chéo khớp với tần số KỲ VỌNG nếu không có liên hệ gì giữa chúng.

In [ ]:
ct1 = pd.crosstab(df["gender"], df["method"])
display(ct1)
chi2, p, dof, exp = stats.chi2_contingency(ct1, correction=True)
print(f"chi2({dof})={chi2:.2f}, p={p:.3f}  (đã hiệu chỉnh Yates cho bảng 2x2)")

#### 📤 Đầu ra thật
χ²(1)=1,37, p=0,243 → KHÔNG có ý nghĩa. ✅ Khớp đúng bài.

## 2. Chi-square: giới tính × trường (bảng 2×3) + Cramer's V

🎯 **Phương pháp này trả lời câu hỏi gì?** Cramer's V đo ĐỘ MẠNH của liên hệ giữa 2 biến định danh (tương tự vai trò của |r| trong tương quan Pearson) — chạy TỪ 0 (không liên hệ) TỚI 1 (liên hệ hoàn hảo). Khác với p-value (chỉ nói CÓ/KHÔNG có ý nghĩa thống kê), Cramer's V cho biết liên hệ đó MẠNH tới mức nào — 2 câu hỏi khác nhau, giống bài học ở Module 04 (ý nghĩa thống kê ≠ độ lớn thực tế).

In [ ]:
ct2 = pd.crosstab(df["gender"], df["school"])
display(ct2)
chi2b, pb, dofb, expb = stats.chi2_contingency(ct2, correction=False)
n_total = ct2.values.sum()
cramer_v = np.sqrt(chi2b / (n_total * (min(ct2.shape)-1)))
pct_small = (expb < 5).mean() * 100
print(f"chi2({dofb})={chi2b:.2f}, p={pb:.3f}, Cramer's V={cramer_v:.3f}")
print(f"% ô có tần số kỳ vọng <5: {pct_small:.0f}% (quy tắc an toàn: phải <20%)")

#### 📤 Đầu ra thật
χ²(2)=0,14, p=0,932, Cramer's V≈0,024 (gần như 0), 0% ô vi phạm quy tắc 20% → kết quả đáng tin cậy, không có ý nghĩa.

## 3. Mann-Whitney U: đánh giá `h1` theo giới tính

🎯 **Phương pháp này trả lời câu hỏi gì?** Mann-Whitney U là phiên bản phi tham số của t-test độc lập: kiểm tra liệu MỘT trong 2 nhóm có xu hướng cho ra giá trị CAO HƠN nhóm kia hay không — dựa trên XẾP HẠNG toàn bộ quan sát (gộp cả 2 nhóm), không dựa trên trung bình. Dùng khi dữ liệu là thứ bậc (như thang Likert) hoặc vi phạm giả định phân phối chuẩn của t-test.

In [ ]:
h1_nam = df.loc[df["gender"]=="Nam", "h1"].dropna()
h1_nu = df.loc[df["gender"]=="Nữ", "h1"].dropna()
u, p = stats.mannwhitneyu(h1_nam, h1_nu)
print(f"U={u:.1f}, p={p:.3f}")

#### 📤 Đầu ra thật
U=6416, p=0,127 → không có ý nghĩa.

## 4. Wilcoxon: so sánh 2 mục đánh giá `a1` và `a2` (cùng 240 học sinh)

🎯 **Phương pháp này trả lời câu hỏi gì?** Wilcoxon là phiên bản phi tham số của t-test bắt cặp: kiểm tra liệu độ CHÊNH LỆCH giữa 2 lần đo trên CÙNG đối tượng có xu hướng lệch về một phía (dương hoặc âm) hay không — dựa trên XẾP HẠNG độ lớn của các chênh lệch, không dựa trên giá trị trung bình.

In [ ]:
w_stat, p_w = stats.wilcoxon(df["a1"], df["a2"])
print(f"W={w_stat:.1f}, p={p_w:.3f}")
print(f"Mean a1={df['a1'].mean():.3f}, Mean a2={df['a2'].mean():.3f}")

#### 📤 Đầu ra thật
p=0,920 — hoàn toàn KHÔNG có khác biệt (TB a1≈2,908, a2≈2,904, rất gần nhau).

## 5. Kruskal-Wallis: đánh giá `h1` theo 3 trường

🎯 **Phương pháp này trả lời câu hỏi gì?** Kruskal-Wallis là phiên bản phi tham số của ANOVA một chiều: kiểm tra 3+ nhóm ĐỘC LẬP có phân phối THỨ HẠNG khác nhau hay không — dùng khi dữ liệu là thứ bậc hoặc vi phạm giả định phân phối chuẩn của ANOVA. Giống ANOVA, kết quả có ý nghĩa KHÔNG cho biết nhóm nào khác biệt.

In [ ]:
groups_h1 = [g["h1"].dropna().values for _, g in df.groupby("school")]
h_stat, p_kw = stats.kruskal(*groups_h1)
print(f"H={h_stat:.3f}, p={p_kw:.3f}")

#### 📤 Đầu ra thật
H=2,844, p=0,241 — không có ý nghĩa. Khác với kết quả có ý nghĩa (F=5,87, p=0,003) khi so sánh điểm Toán theo trường ở Module 06 — hai biến khác nhau, hai câu trả lời khác nhau.

## 6. Friedman: 3 mục đánh giá `a1`, `a2`, `a3` (cùng 240 học sinh)

🎯 **Phương pháp này trả lời câu hỏi gì?** Friedman là phiên bản phi tham số của ANOVA đo lường lặp lại: kiểm tra 3+ lần đo/biến trên CÙNG một nhóm đối tượng có xu hướng XẾP HẠNG khác nhau hay không (dựa trên thứ hạng trong từng đối tượng, không dựa trên giá trị trung bình thô).

In [ ]:
chi2_f, p_f = stats.friedmanchisquare(df["a1"], df["a2"], df["a3"])
print(f"chi2={chi2_f:.3f}, p={p_f:.3f}")

#### 📤 Đầu ra thật
χ²=0,555, p=0,758 — không có ý nghĩa, nhóm đánh giá các khía cạnh liên quan khá đồng đều.